In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd

# Ingest raw customer data from a CSV file
raw_data = pd.read_csv('customer_data.csv')

# Clean the data
# 1. Standardize column names
raw_data.columns = [col.strip().lower().replace(' ', '_') for col in raw_data.columns]

# 2. Remove duplicates
cleaned_data = raw_data.drop_duplicates()

# 3. Handle missing values (example: fill with empty string or drop rows)
# For demonstration, fill missing values with empty string
cleaned_data = cleaned_data.fillna('')

# 4. Basic data type transformations (example: ensure 'customer_id' is string)
if 'customer_id' in cleaned_data.columns:
    cleaned_data['customer_id'] = cleaned_data['customer_id'].astype(str)

# Save cleaned data for next steps
cleaned_data.to_csv('customer_data_cleaned.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'customer_data.csv'

In [3]:
import pandas as pd

# Step 1: Generate sample raw customer data since 'customer_data.csv' does not exist
sample_data = {
    'Customer ID': [101, 102, 103, 103, None],
    'Name': ['Alice Smith', 'Bob Jones', 'Charlie Lee', 'Charlie Lee', 'Dana White'],
    'Email': ['alice@example.com', 'bob@example.com', None, None, 'dana@example.com'],
    'Signup Date': ['2023-01-01', '2023-01-05', '2023-01-07', '2023-01-07', '2023-01-10']
}
raw_data = pd.DataFrame(sample_data)

# Step 2: Standardize column names
raw_data.columns = [col.strip().lower().replace(' ', '_') for col in raw_data.columns]

# Step 3: Remove duplicates
cleaned_data = raw_data.drop_duplicates()

# Step 4: Handle missing values (fill with empty string for demonstration)
cleaned_data = cleaned_data.fillna('')

# Step 5: Basic data type transformations (ensure 'customer_id' is string)
if 'customer_id' in cleaned_data.columns:
    cleaned_data['customer_id'] = cleaned_data['customer_id'].astype(str)

# Step 6: Save cleaned data for next steps
cleaned_data.to_csv('customer_data_cleaned.csv', index=False)

# Display cleaned data for verification
print(cleaned_data)

  customer_id         name              email signup_date
0       101.0  Alice Smith  alice@example.com  2023-01-01
1       102.0    Bob Jones    bob@example.com  2023-01-05
2       103.0  Charlie Lee                     2023-01-07
4               Dana White   dana@example.com  2023-01-10


In [4]:
import sqlite3
import pandas as pd

# Load cleaned data
df = pd.read_csv('customer_data_cleaned.csv')

# Connect to SQLite database (simulating a data warehouse)
conn = sqlite3.connect('customer_warehouse.db')
cursor = conn.cursor()

table_name = 'customers'

# Get current columns in the database table (if exists)
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone() is not None

if table_exists:
    cursor.execute(f"PRAGMA table_info({table_name});")
    db_columns = [row[1] for row in cursor.fetchall()]
else:
    db_columns = []

csv_columns = list(df.columns)

# Schema change handling: add new columns if any
if table_exists:
    for col in csv_columns:
        if col not in db_columns:
            cursor.execute(f"ALTER TABLE {table_name} ADD COLUMN {col} TEXT;")
    conn.commit()
else:
    # Create table with all columns as TEXT for simplicity
    columns_def = ', '.join([f"{col} TEXT" for col in csv_columns])
    cursor.execute(f"CREATE TABLE {table_name} ({columns_def});")
    conn.commit()

# Data quality checks
quality_report = {}

# 1. Uniqueness of customer_id
if 'customer_id' in df.columns:
    unique_count = df['customer_id'].nunique()
    total_count = len(df)
    quality_report['customer_id_unique'] = (unique_count == total_count)

# 2. Null value detection
quality_report['nulls_per_column'] = df.isnull().sum().to_dict()

# 3. Data type consistency (all columns should be string/object)
quality_report['dtype_consistency'] = all(df[col].dtype == object or df[col].dtype == 'O' for col in df.columns)

# Insert data into the table (replace existing for demo)
cursor.execute(f"DELETE FROM {table_name};")
df.to_sql(table_name, conn, if_exists='append', index=False)

conn.commit()
conn.close()

print("Data loaded into warehouse. Data quality report:")
print(quality_report)

Data loaded into warehouse. Data quality report:
{'customer_id_unique': False, 'nulls_per_column': {'customer_id': 1, 'name': 0, 'email': 1, 'signup_date': 0}, 'dtype_consistency': False}


In [5]:
import sqlite3
import re

class SemanticLayer:
    def __init__(self, db_path='customer_warehouse.db', table_name='customers'):
        self.db_path = db_path
        self.table_name = table_name

    def _nl_to_sql(self, nl_query: str) -> str:
        # Basic intent mapping
        nl_query = nl_query.lower()
        if "list all customers" in nl_query or "show all customers" in nl_query:
            return f"SELECT * FROM {self.table_name};"
        match = re.search(r"signed up after (\d{4}-\d{2}-\d{2})", nl_query)
        if match:
            date = match.group(1)
            return f"SELECT * FROM {self.table_name} WHERE signup_date > '{date}';"
        match = re.search(r"search for (.+)", nl_query)
        if match:
            name = match.group(1).strip()
            return f"SELECT * FROM {self.table_name} WHERE name LIKE '%{name}%';"
        match = re.search(r"customers named ([\w\s]+)", nl_query)
        if match:
            name = match.group(1).strip()
            return f"SELECT * FROM {self.table_name} WHERE name LIKE '%{name}%';"
        # Default: return all
        return f"SELECT * FROM {self.table_name};"

    def query(self, nl_query: str):
        sql = self._nl_to_sql(nl_query)
        conn = sqlite3.connect(self.db_path)
        df = None
        try:
            df = pd.read_sql_query(sql, conn)
        finally:
            conn.close()
        return df

# Example usage:
semantic = SemanticLayer()
result = semantic.query("Show all customers who signed up after 2023-01-05")
print(result)

  customer_id         name              email signup_date
0       101.0  Alice Smith  alice@example.com  2023-01-01
1       102.0    Bob Jones    bob@example.com  2023-01-05
2       103.0  Charlie Lee               None  2023-01-07
3        None   Dana White   dana@example.com  2023-01-10
